In [1]:
"""
Cell2location spatial deconvolution
Reference  : integrated_annot_fine.h5ad  (annot_fine column)
Spatial    : spatial_adata_xenium_cosmx_zenodo.h5ad
"""

import warnings
warnings.filterwarnings("ignore")

import os, numpy as np, pandas as pd, scanpy as sc, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cell2location
from cell2location.models import RegressionModel, Cell2location


# ── Configuration ──────────────────────────────────────────────────────────────
DATA_DIR   = r"C:\Users\RICHMOND\Documents\Richmond\singlecell_rein\script_exemple\data2\Annalyse vrai"
RESULTS    = os.path.join(DATA_DIR, "cell2location_results")
os.makedirs(RESULTS, exist_ok=True)

# Set TEST_MODE=True to run on a 100K-cell subset (fast validation on CPU).
# Set to False for the full 4.3M-cell run (hours on CPU).
TEST_MODE  = True
N_TEST     = 100_000          # cells sampled from spatial for test run
REF_EPOCHS = 250              # epochs for NB regression reference model
C2L_EPOCHS = 30_000           # max epochs for cell2location (uses early stopping)
BATCH_SIZE = 2_048            # minibatch size (increase if RAM allows)

# For single-cell resolution spatial (Xenium / CosMx) each spot ≈ 1 cell
N_CELLS_PER_LOCATION = 1
DETECTION_ALPHA      = 20     # typical for Xenium/CosMx

print(f"TEST_MODE = {TEST_MODE}")
print(f"Results   -> {RESULTS}")

# ── 1. Load data ───────────────────────────────────────────────────────────────
print("\n[1/6] Loading scRNA reference ...")
sc_adata = sc.read_h5ad(os.path.join(DATA_DIR, "integrated_annot_fine.h5ad"))

print("[1/6] Loading spatial data ...")
sp_adata = sc.read_h5ad(os.path.join(DATA_DIR, "spatial_adata_xenium_cosmx_zenodo.h5ad"))

# ── 2. Prepare scRNA reference ─────────────────────────────────────────────────
print("\n[2/6] Preparing scRNA reference ...")

import scipy.sparse

sc_ref = sc_adata.copy()

# layers['counts'] stores log-normalized values (raw counts were lost during
# integration). Reverse with expm1 + round to recover pseudo-counts as integers.
raw = sc_ref.layers["counts"]
if scipy.sparse.issparse(raw):
    pseudo = raw.copy().astype(np.float32)
    pseudo.data = np.round(np.expm1(pseudo.data)).astype(np.float32)
    pseudo.data = np.clip(pseudo.data, 0, None)
else:
    pseudo = np.round(np.expm1(raw)).astype(np.float32)
    pseudo = np.clip(pseudo, 0, None)
sc_ref.X = pseudo
print("  Pseudo-counts: expm1(counts) rounded to integers")

# Remove cells without annotation
sc_ref = sc_ref[sc_ref.obs["annot_fine"] != "Unassigned"].copy()
print(f"  scRNA cells after removing Unassigned: {sc_ref.n_obs:,}")
print(f"  Cell types: {sc_ref.obs['annot_fine'].nunique()}")

# Filter to genes also present in spatial data
common_genes = sc_ref.var.index.intersection(sp_adata.var.index)
print(f"  Common genes: {len(common_genes)}")
sc_ref = sc_ref[:, common_genes].copy()

# Cell2location gene QC: keep genes detected in enough cells
cell2location.utils.filtering.filter_genes(
    sc_ref,
    cell_count_cutoff=5,
    cell_percentage_cutoff2=0.03,
    nonz_mean_cutoff=1.12,
)
print(f"  Genes after QC: {sc_ref.n_vars}")

# ── 3. Estimate reference cell type signatures (NB regression) ─────────────────
print("\n[3/6] Training NB regression reference model ...")

RegressionModel.setup_anndata(
    sc_ref,
    labels_key="annot_fine",
    batch_key="dataset",
)

reg_model = RegressionModel(sc_ref)
reg_model.view_anndata_setup()

reg_model.train(
    max_epochs=REF_EPOCHS,
    accelerator="cpu",
    batch_size=BATCH_SIZE,
)

# Save training loss curve
fig = reg_model.plot_history(20)
plt.savefig(os.path.join(RESULTS, "ref_model_loss.png"), dpi=120, bbox_inches="tight")
plt.close()

# Export estimated expression signatures
sc_ref = reg_model.export_posterior(
    sc_ref,
    sample_kwargs={"num_samples": 1000, "batch_size": BATCH_SIZE},
)

# Save reference model
reg_model.save(os.path.join(RESULTS, "ref_model"), overwrite=True)

# Extract per-cell-type signatures (mean expression per gene per cell type)
if "means_per_cluster_mu_fg" in sc_ref.varm:
    inf_aver = sc_ref.varm["means_per_cluster_mu_fg"][
        [f for f in sc_ref.varm["means_per_cluster_mu_fg"].columns
         if "means_per_cluster_mu_fg" in f]
    ].copy()
else:
    inf_aver = sc_ref.varm["means_per_cluster_mu_fg"].copy()

inf_aver.columns = [c.replace("means_per_cluster_mu_fg_", "") for c in inf_aver.columns]
inf_aver.to_csv(os.path.join(RESULTS, "cell_type_signatures.csv"))
print(f"  Signatures shape: {inf_aver.shape}")

# ── 4. Prepare spatial data ────────────────────────────────────────────────────
print("\n[4/6] Preparing spatial data ...")

sp = sp_adata.copy()
sp.X = sp.layers["counts"].copy()

# Subset to common genes (those that passed QC in reference)
shared = sc_ref.var.index.intersection(sp.var.index)
sp = sp[:, shared].copy()
print(f"  Spatial genes after QC filter: {sp.n_vars}")

if TEST_MODE:
    np.random.seed(42)
    idx = np.random.choice(sp.n_obs, size=min(N_TEST, sp.n_obs), replace=False)
    sp = sp[idx].copy()
    print(f"  TEST_MODE: subsampled to {sp.n_obs:,} cells")
else:
    print(f"  Full mode: {sp.n_obs:,} cells (this will take several hours on CPU)")

# ── 5. Train cell2location ─────────────────────────────────────────────────────
print("\n[5/6] Training cell2location model ...")

Cell2location.setup_anndata(sp, batch_key="orig_ident")

c2l_model = Cell2location(
    sp,
    cell_state_df=inf_aver,
    N_cells_per_location=N_CELLS_PER_LOCATION,
    detection_alpha=DETECTION_ALPHA,
)
c2l_model.view_anndata_setup()

c2l_model.train(
    max_epochs=C2L_EPOCHS,
    batch_size=BATCH_SIZE,
    accelerator="cpu",
    early_stopping=True,
)

# Save training loss curve
fig = c2l_model.plot_history(1000)
plt.savefig(os.path.join(RESULTS, "c2l_model_loss.png"), dpi=120, bbox_inches="tight")
plt.close()

# Export posterior (estimated cell abundances per spot)
sp = c2l_model.export_posterior(
    sp,
    sample_kwargs={
        "num_samples": 1000,
        "batch_size": BATCH_SIZE,
    },
)

# Save cell2location model
c2l_model.save(os.path.join(RESULTS, "c2l_model"), overwrite=True)

# ── 6. Extract results & visualize ────────────────────────────────────────────
print("\n[6/6] Extracting results and plotting ...")

# Cell abundance matrix (5% quantile — conservative estimate)
if "q05_cell_abundance_w_sf" in sp.obsm:
    abund = pd.DataFrame(
        sp.obsm["q05_cell_abundance_w_sf"],
        index=sp.obs_names,
        columns=inf_aver.columns,
    )
else:
    abund = pd.DataFrame(
        sp.obsm["means_cell_abundance_w_sf"],
        index=sp.obs_names,
        columns=inf_aver.columns,
    )

abund.to_csv(os.path.join(RESULTS, "cell_abundances.csv.gz"), compression="gzip")
print(f"  Abundance table: {abund.shape}")

# Add abundances back to AnnData obs
sp.obs = sp.obs.join(abund.add_prefix("c2l_"), how="left")

# ── Plot spatial maps for each cell type ──────────────────────────────────────
cell_types = inf_aver.columns.tolist()
coords = sp.obsm["spatial"]
n_cols = 5
n_rows = int(np.ceil(len(cell_types) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3.5))
axes = axes.flatten()

for i, ct in enumerate(cell_types):
    col = f"c2l_{ct}"
    if col not in sp.obs.columns:
        continue
    vals = sp.obs[col].values
    vmax = np.percentile(vals, 99)
    sc_ = axes[i].scatter(
        coords[:, 0], coords[:, 1],
        c=vals, cmap="inferno", s=0.3, vmin=0, vmax=vmax,
        rasterized=True,
    )
    axes[i].set_title(ct, fontsize=7, fontweight="bold")
    axes[i].axis("off")
    plt.colorbar(sc_, ax=axes[i], fraction=0.046, pad=0.02)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Cell2location — cell type abundances (q05)", fontsize=14, y=1.01)
plt.tight_layout()
out_fig = os.path.join(RESULTS, "spatial_cell_types.png")
plt.savefig(out_fig, dpi=120, bbox_inches="tight")
plt.close()
print(f"  Spatial map saved: {out_fig}")

# ── Save updated spatial AnnData ──────────────────────────────────────────────
suffix = "_test" if TEST_MODE else "_full"
out_sp = os.path.join(RESULTS, f"spatial_c2l{suffix}.h5ad")
sp.write_h5ad(out_sp, compression="gzip")
print(f"  Spatial AnnData saved: {out_sp}")

print("\n=== Done ===")
print(f"Results in: {RESULTS}")


TEST_MODE = True
Results   -> C:\Users\RICHMOND\Documents\Richmond\singlecell_rein\script_exemple\data2\Annalyse vrai\cell2location_results

[1/6] Loading scRNA reference ...
[1/6] Loading spatial data ...

[2/6] Preparing scRNA reference ...
  Pseudo-counts: expm1(counts) rounded to integers
  scRNA cells after removing Unassigned: 131,978
  Cell types: 44
  Common genes: 1211
  Genes after QC: 1211

[3/6] Training NB regression reference model ...


Anndata setup with scvi-tools version 1.4.2.

Setup via `RegressionModel.setup_anndata` with arguments:

{
│   'layer': None,
│   'batch_key': 'dataset',
│   'labels_key': 'annot_fine',
│   'categorical_covariate_keys': None,
│   'continuous_covariate_keys': None
}

         Summary Statistics          
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃     Summary Stat Key     ┃ Value  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│         n_batch          │   2    │
│         n_cells          │ 131978 │
│ n_extra_categorical_covs │   0    │
│ n_extra_continuous_covs  │   0    │
│         n_labels         │   44   │
│          n_vars          │  1211  │
└──────────────────────────┴────────┘

...

               Data Registry                
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Registry Key ┃    scvi-tools Location    ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      X       │          adata.X          │
│    batch     │ adata.obs['_scvi_batch']  │
│    ind_x     │   adata.obs['_indices']   │
│    labels    │ adata.obs['_scvi_labels'] │
└──────────────┴───────────────────────────┘

                     batch State Registry                     
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃     Source Location     ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['orig_ident'] │    1001    │          0          │
│                         │    1003    │          1          │
│                         │    1004    │          2          │
│                         │    1005    │          3          │
│                         │    1006    │          4          │
│                         │    1007    │          5          │
│                         │    1008    │          6          │
│                         │    1009    │          7          │
│                         │    1010    │          8          │
│                         │    1011    │          9          │
│                         │    1012    │         10          │
│                         │    1013    │         11          │
│                         │    1061    │         12          │
│                         │    1062    │         13          │
│                         │    1063    │         14          │
│                         │    1064    │         15          │
│                         │    1068    │         16          │
│                         │    1070    │         17          │
│                         │    1071    │         18          │
│                         │    1072    │         19          │
│                         │   HK2695   │         20          │
│                         │   HK2753   │         21          │
│                         │   HK2841   │         22          │
│                         │   HK2844   │         23          │
│                         │  HK2844_2  │         24          │
│                         │   HK2873   │         25          │
│                         │   HK2874   │         26          │
│                         │   HK2924   │         27          │
│                         │   HK2989   │         28          │
│                         │   HK2990   │         29          │
│                         │   HK3035   │         30          │
│                         │  HK3035_2  │         31          │
│                         │   HK3039   │         32          │
│                         │   HK3063   │         33          │
│                         │   HK3066   │         34          │
│                         │   HK3068   │         35          │
│                         │   HK3069   │         36          │
│                         │   HK3070   │         37          │
│                         │   HK3106   │         38          │
│                         │   HK3421   │         39          │
│                         │   HK3469   │         40          │
│                         │   HK3474   │         41          │
│                         │   HK3531   │         42          │
│                         │  HK3531_2  │         43          │
│                         │   HK3535   │         44          │
│                         │   HK3542   │         45          │
│                         │   HK3565   │         46          │
│                         │   HK3588   │         47          │
│                         │   HK3591   │         48          │
│                         │   HK3594   │         49          │
│                         │   HK3606   │         50          │
│                         │   HK3612   │         51          │
│                         │   HK3614   │         52          │
│                         │   HK3616   │         53          │
│                         │   HK3623   │         54          │
│                         │   HK3624   │         55          │
│                         │   HK3626   │         56          │
│                         │   HK3631   │         57          │
│                         │   HK3647   │         58          │
│                         │ Ped

                     labels State Registry                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃      Source Location      ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['_scvi_labels'] │     0      │          0          │
└───────────────────────────┴────────────┴─────────────────────┘

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


ValueError: Can't run Early Stopping with validation monitor with no validation set